# Py Cost Tools
A suite of tools to help cost estimators

## Installation
* requires a copy of git 
* otherwise need to clone / copy to desktop then run setup.py

```
pip install git+https://github.com/frankij11/PyCostTools.git#egg=pycost
```

## Inflation
* use the inflation functions as a simple calculator

In [ ]:
import pycost as ct
import warnings
warnings.filterwarnings("ignore")

print("BY20 to BY25", ct.inflation.BYtoBY(Index = "APN", FromYR = 2020, ToYR = 2025, Cost = 1))
print("TY25 to BY20", ct.inflation.TYtoBY(Index = "APN", FromYR = 2025, ToYR = 2020, Cost = 1))
print("Multiple Values", ct.inflation.BYtoBY(Index =['APN',"APN",'APN'], FromYR = [2020,2021,2022], ToYR = 2023, Cost=1))

* or use the inflation function as part of some analysis
* works well with DataFrames

In [ ]:
import pandas as pd
df = pd.DataFrame({'Index': ['APN']*10, 'Fiscal_Year':range(2020,2030), 'BY20': 1 })
df

In [ ]:
df.assign(TY_DOL = lambda x: ct.inflation.BYtoTY(Index = x.Index,FromYR = 2020, ToYR=x.Fiscal_Year, Cost = x.BY20))

## Analysis
* helper functions to build models
* comes preloaded with several sklearn models

As an example let's use data from the Joint Inflation Calculator to predict inflation

### Toy Example: basics model flow
1. Define Model with data, formula, model (other specifications available)
1. Fit Model
1. View Summary
1. View Report
1. Make Predications
1. Save Model for later

In [ ]:
# Import Libraries
from pycost.analysis import Model, Models, AutoRegressionLinear
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV

In [ ]:
df = pd.DataFrame({'y': [1.5,2.2,3.2,4.9,5.0], 'x1': [2,4,6,8,10], 'x2': ["a", "b","b","a","a"]})
myModel = Model(df, "y~x1", model= LinearRegression(),test_split=0,
        meta_data={
            'title': "Example Analysis",
            'desc': "Do some anlaysis",
            'analyst': 'Kevin Joy',
            'FreeFileds': "Make whatever you like to doucment analysis"}
            )
myModel.fit().summary()

In [ ]:
# show interactive report with fit statistics
#myModel_report = myModel.report(False)
#myModel_report

## Many Models: Motivating example
* Imagine having a dateset where we want to do a regression for value in each category columns
* We could filter dataset and run regression for each column
* or better yet use the groupby function for pandas

In [ ]:
df = ct.inflation.jic.assign(Year=lambda x: pd.to_numeric(x.Year, 'coerce') ) # The year variable in jic is read as a string so must be converted
apn_df = df.query('Indice =="APN"')

In [ ]:
apnModel = Model(apn_df, "Raw~Year", model=LinearRegression(),test_split=.4, handle_na=False)
apnModel.fit()
apnModel.summary()

In [ ]:
#apnModel.report(False)

* we can use the Models API to run multiple formulas for each value in a given category

In [ ]:
manyModels = Models(df,formulas=["Raw~Year"],by=['Service', "Indice"], test_split=-1, handle_na=False)
manyModels.fit()

In [ ]:
manyModels.db

In [ ]:
manyModels.summary()

 * Now imagine you want to build more models and add to your database

In [ ]:
manyModels.build_models(df, "Raw ~ Year + Indice-1")

In [ ]:
apn_Models = manyModels.db.query('Indice =="APN"')
apn_Models

In [ ]:
apn_Models.Model.to_list()[0].report()

In [ ]:
autoModel = AutoRegressionLinear(n_iter=10)
autoModel.fit(X=df.drop('Raw',axis=1), y=df.Raw)
autoModel.summary()

In [ ]:
autoModel.best_estimator_

In [ ]:
df.assign(
    Raw_Predictions = autoModel.predict(df.drop('Raw', axis=1)),
    Raw_Errors = lambda x: x.Raw - x.Raw_Predictions
).plot(x='Year', y='Raw_Errors',kind='scatter')

In [ ]:
!pip install --upgrade param

In [ ]:
import param
import pandas as pd
from pycost.cost_estimate.core.base import Model
from pycost.cost_estimate.core.parent_models import ParentModel
from pycost.cost_estimate.utils import reactive
#from pycost.cost_estimate.utils.sim_tool import SimEgine

In [ ]:
print(param.__version__)
class GlobalInputs(param.Parameterized):
    base_year = param.Number(2020)
global_inputs = GlobalInputs()
class Labor(Model):
    hours = param.Number(100)
    labor_rate = param.Number(55, doc='dollars per hour')
    base_year = param.Number(global_inputs.param.base_year, allow_refs=True)
    base_yearG = param.Number(GlobalInputs.param.base_year, allow_refs=True)
    
    @param.depends('hours', 'labor_rate', watch=True)
    def calc_cost(self):

        self.cost = pd.DataFrame(dict(FY=[2020], value_cp = [self.hours * self.labor_rate]))

        return self.cost
    @param.depends('base_year', watch=True)
    def update_base_year(self):
        print('updating instance')
        print("instance:", self.base_year)
        print("global:", self.base_yearG)
    @param.depends('base_yearG', watch=True)
    def update_by_g(self):
        print('updating instance')
        print("instance:", self.base_year)
        print("global:", self.base_yearG)
                              
        

In [ ]:
l = Labor()
l.hours = 20

In [ ]:
l.uncertainty = 1.1
l.cost_estimate_metadata

In [ ]:
import param

In [ ]:
param.__version__

In [ ]:
print(param.__version__)
class GlobalInputs(param.Parameterized):
    base_year = param.Number(2020)
    n = param.Number(30)
global_inputs = GlobalInputs()
class Labor(param.Parameterized):
    hours = param.Number(100)
    labor_rate = param.Number(55, doc='dollars per hour')
    base_year = param.Number(global_inputs.base_year, allow_refs=True)
    base_yearG = param.Parameter(global_inputs, allow_refs=True, instantiate=False)
    
    @param.depends('hours', 'labor_rate', watch=True)
    def calc_cost(self):

        self.cost = pd.DataFrame(dict(FY=[2020], value_cp = [self.hours * self.labor_rate]))

        return self.cost
    @param.depends('base_year', watch=True)
    def update_base_year(self):
        print('updating instance')
        print("instance:", self.base_year)
        print("global:", self.base_yearG)
    @param.depends('base_yearG.base_year', watch=True)
    def update_by_g(self):
        print('updating global')
        print("instance:", self.base_year)
        print("global:", self.base_yearG)
                              
        

In [ ]:
l = Labor(base_year=global_inputs.param.base_year)
l

In [ ]:
global_inputs.base_year=2025

In [ ]:
l.base_year = global_inputs.param.base_year

In [ ]:
l.base_yearG.base_year=2025

In [ ]:
global_inputs

In [ ]:
import param
import pandas as pd

class GlobalInputs(param.Parameterized):
    """
    Global parameters and settings for cost estimation.
    
    This class defines program-wide parameters that affect all cost models,
    including program identification, base year, and currency units.
    
    Attributes:
        long_name (str): Full program name
        short_name (str): Program abbreviation
        base_year (int): Base year for cost calculations (1970-2060)
        dol_units (int): Currency unit multiplier (1, 1K, 1M, 1B)
        required_fields (List[str]): Required fields in cost estimates
        report_fields (List[str]): Fields to include in reports
    """
    
    long_name = param.String("Program X")
    short_name = param.String("X")
    base_year = param.Integer(2020, bounds=(1970, 2060))
    dol_units = param.Selector(objects=[1, 1_000, 1_000_000, 1_000_000_000], default=1_000)
    required_fields = param.List(['FY', 'value_cp'])
    report_fields = param.List(['model', 'appn', 'FY', 'value_cp', 'value_ty', 'value_cy'])

    @property
    def BY(self) -> int:
        """Return the base year for calculations."""
        return self.base_year
    
    def __panel__(self):
        """Create a Panel interface for the parameters."""
        return pn.Column(self.param)


class Model(param.Parameterized): #
    """
    Base class for cost estimation models.
    
    This class provides the foundation for all cost estimation models,
    implementing reactive parameter management and calculation updates.
    
    Attributes:
        meta (Dict): Metadata about the model (analyst, element, etc.)
        global_inputs (GlobalInputs): Program-wide parameters
        uncertainty_inputs (Dict): Uncertainty parameters
        uncertainty (float): Uncertainty factor
        sim_results (pd.DataFrame): Simulation results
        cost_estimate (pd.DataFrame): The calculated cost estimate
        schedule_estimate (pd.DataFrame): The calculated schedule estimate
        total_cost_cp (float): Total constant price cost
        total_cost_ty (float): Total then-year cost
    """
    
    meta = param.Dict(
        default={
            'Analyst': "N/A",
            'Element': "N/A",
        }
    )
    global_inputs = param.ClassSelector(GlobalInputs, GlobalInputs(), instantiate=False)
    uncertainty_inputs = param.Dict(default={
                'uncertainty': 1
            })
    uncertainty = param.Number(1)
    simulate = param.Action(lambda x: x.run_simulation(100))
    sim_results = param.DataFrame(precedence=.1)
    cost_estimate = param.DataFrame(precedence=.1)
    schedule_estimate = param.DataFrame(precedence=.1)
    total_cost_cp = param.Number(precedence=.1, constant=True)
    total_cost_ty = param.Number(precedence=.1, constant=True)
  

    
    def __call__(self, update: bool = True, **params) -> pd.DataFrame:
        """
        Execute the model calculations.
        
        Args:
            update: Whether to update parameters before calculation
            **params: Parameters to update
            
        Returns:
            The cost estimate DataFrame
        """
        if update:
            self.param.update(**params)
        else:
            pass
            #logger.warning("Temporary update not implemented yet")
        return self.cost_estimate

    @property
    def parent_path(self) -> str:
        """Get the parent path of the model."""
        if hasattr(self, "_parent_path"):
            return self._parent_path
        else:
            self._parent_path = self.__class__.__name__
        return self._parent_path
    
    @parent_path.setter
    def parent_path(self, value: str) -> None:
        """Set the parent path of the model."""
        self._parent_path = value

    @property
    def level(self) -> int:
        """Get the model's hierarchy level."""
        level = self.parent_path.count('>') + 1
        return level

    def calc_cost(self) -> pd.DataFrame:
        """
        Calculate the base cost estimate.
        
        This method should be implemented by subclasses to provide
        specific cost calculation logic.
        
        Returns:
            DataFrame containing the cost estimate
        """
        #logger.warning("calc_cost is not implemented")
        print("calc_cost")
        self.cost_estimate = pd.DataFrame(columns=self.global_inputs.required_fields)
        return self.cost_estimate
    def calc_sched(self):
        self.schedule_estimate = pd.DataFrame([1,2,34])
        return
    
    @param.depends('calc_cost', 'uncertainty', watch=True)
    def calc_cost_uncertainty(self) -> pd.DataFrame:
        """
        Apply uncertainty factors to the cost estimate.
        
        Returns:
            DataFrame with uncertainty-adjusted costs
        """
        #logger.debug("Applying uncertainty factors")
        print('update uncertainty')
        self._cost_estimate = self.cost_estimate.copy()
        self._cost_estimate= self.cost_estimate.assign(
            uncertainty=self.uncertainty,
            value_cp=lambda x: x.value_cp * self.uncertainty
        )
        #self.param.cost_estimate = self._cost_estimate
        return self._cost_estimate
    
    @param.depends('calc_cost_uncertainty', 'global_inputs.base_year',watch=True)
    def calc_cost_metadata(self) -> pd.DataFrame:
        """
        Create the final cost estimate with model information.
        
        Returns:
            DataFrame containing the final cost estimate
        """
        print('updating meta')
        df = self.cost_estimate.copy()
        # initialize to required fields
        col_order = []

        #logger.debug("Creating final cost estimate")
        # add required fields to the cost estimate
        
        missing_fields = set(self.global_inputs.required_fields+self.global_inputs.report_fields) - set(df.columns)
        if len(missing_fields) > 0:
            #logger.debug(f"Missing fields: {missing_fields}")
            df = df.assign(
                **{field: None for field in missing_fields}
            )
        # add parent path and level to the cost estimate
        path_list = self.parent_path.split(' > ')
        for i, path in enumerate(path_list):
            col_order.append("level " + str(i))
            df = df.assign(
                **{"level " + str(i): path}
            )
        # add this model's name
        col_order.append("model")
        
        df = df.assign(
            **{"model": self.__class__.__name__}
        )
        # add cp and ty columns
        col_order.append('FY')
        col_order.append("value_cp")
        col_order.append("value_ty")
        col_order.append('value_cy')
        col_order.append('global_base_year')
        if 'base_year' not in df.columns:
            df['base_year'] = self.global_inputs.base_year
        df['value_cp'] = df['value_cp'] # To Do units
        print('  cp')
        df['value_ty'] = df['value_cp'] # To DO escalation
        print('  ty')
        df['value_cy'] = df['value_ty'] # To DO inflation
        print(' cy')
        # get meta columns
        for key, val in self.meta.items():
            col_order.append("meta_" + key)
            df = df.assign(
                **{"meta_" + key: val}
            )
            print('  added meta', key)
        for col in [col for col in df.columns if col not in col_order]:
            col_order.append(col)
        # reorder columns
        print(col_order)
        df = df[col_order]
        print('  pre assign cost')
        self._cost_estimate = df
        print('  post assign cost')
        #self.param.cost_estimate = self._cost_estimate.copy()
        return self._cost_estimate
    


In [ ]:
class Labor(Model):
    hours = param.Number(100)
    labor_rate = param.Number(55, doc='dollars per hour')
    
    @param.depends('hours', 'labor_rate', watch=True)
    def calc_cost(self):
        print('calc_cost')
        self.cost_estimate = pd.DataFrame(dict(FY=[2020], value_cp = [self.hours * self.labor_rate]))

        return self.cost_estimate
    @param.depends('cost_estimate', watch=True)
    def update_schedule(self):
        print('update schedule')
        self._schedule_estimate = self.cost_estimate.assign(new_col=1)


In [ ]:
l=Labor()

In [ ]:
l.hours=50
l._cost_estimate

,level 0,model,FY,value_cp,value_ty,value_cy,base_year,meta_Analyst,meta_Element,appn
0,Labor,Labor,2020,2750,2750,2750,2020,N/A,N/A,None


In [ ]:
p=l.param['cost_estimate']
import inspect
inspect.getsource(p._on_set)#?
dir(p)

['_SelectorBase__abstract',
 '__class__',
 '__classdoc',
 '__delattr__',
 '__delete__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__set__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '_internal_name',
 '_label',
 '_length_bounds_check',
 '_on_set',
 '_post_setter',
 '_serializers',
 '_set_instantiate',
 '_set_names',
 '_validate',
 '_validate_class_',
 '_validate_value',
 'allow_None',
 'class_',
 'columns',
 'constant',
 'default',
 'deserialize',
 'doc',
 'get_range',
 'instantiate',
 'is_instance',
 'label',
 'name',
 'ordered',
 'owner',
 'per_instance',
 'pickle_default_value',
 'precedence',
 'readonly',
 'rows',
 'schema',
 'serialize',
 'watchers']

In [ ]:
l.calc_cost()

In [ ]:
l.global_inputs.base_year =2020
l._cost_estimate

In [ ]:
l.param.cost_estimate.watchers

{}

In [ ]:
watchers = l.param.watchers
c1 = watchers['cost_estimate']['value'][0]
for key in watchers.keys():
    print(key)
print(c1.fn())
#l.param.method_dependencies?
for p in l.param.method_dependencies("calc_cost_metadata", True):
    print(p.name)

name
meta
global_inputs
uncertainty_inputs
uncertainty
simulate
sim_results
cost_estimate
schedule_estimate
total_cost_cp
total_cost_ty
hours
labor_rate
update uncertainty


AttributeError: 'NoneType' object has no attribute 'assign'

In [ ]:
import pycost as ct

In [ ]:
nx.find_cycle?

Signature: nx.find_cycle(G, source=None, orientation=None)
Docstring:
Returns a cycle found via depth-first traversal.

The cycle is a list of edges indicating the cyclic path.
Orientation of directed edges is controlled by `orientation`.

Parameters
----------
G : graph
    A directed/undirected graph/multigraph.

source : node, list of nodes
    The node from which the traversal begins. If None, then a source
    is chosen arbitrarily and repeatedly until all edges from each node in
    the graph are searched.

orientation : None | 'original' | 'reverse' | 'ignore' (default: None)
    For directed graphs and directed multigraphs, edge traversals need not
    respect the original orientation of the edges.
    When set to 'reverse' every edge is traversed in the reverse direction.
    When set to 'ignore', every edge is treated as undirected.
    When set to 'original', every edge is treated as directed.
    In all three cases, the yielded edge tuples add a last entry to
    indicate t